In [ ]:
import cv2
import face_recognition
import pickle
import json
import numpy as np
import time

# 1. Encodings aur Details load karein
data = pickle.loads(open("encodings.pickle", "rb").read())
with open("Details.json", "r") as f:
    user_details = json.load(f)
# '0' ki jagah IP Webcam ka link daalo

video_capture = cv2.VideoCapture(0)

# HD Resolution set karein
video_capture.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
video_capture.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

print("HUD System Booting Up...")

while True:
    ret, frame = video_capture.read()
    if not ret:
        break

    # Processing speed ke liye frame resize
    small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
    rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    face_locations = face_recognition.face_locations(rgb_small_frame)
    face_encodings = face_recognition.face_encodings(
        rgb_small_frame, face_locations)

    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
        # jo problem aa rahi ab nahi ajyegi tolerance add karne se agar match 45% se kam perfect hua , toh woh use seedha "ALIEN "declare kar dega
        matches = face_recognition.compare_faces(
            data["encodings"], face_encoding, tolerance=0.5)

        # Default values for Unknown
        name = "Data not found. This person is ALIEN 👽"
        box_color = (0, 0, 255)  # Red

        face_distances = face_recognition.face_distance(
            data["encodings"], face_encoding)
        if len(face_distances) > 0:
            best_match_index = np.argmin(face_distances)
            # distance check:agar distance 0.5 se jayeda hua to identifi nahi karega
            if matches[best_match_index] and face_distances[best_match_index] < 0.6:
                name = data["names"][best_match_index]
                box_color = (0, 255, 0)  # Green for Known

        # Wapas scale up karein
        top, right, bottom, left = top*4, right*4, bottom*4, left*4
        # --- FUTURISTIC HUD DRAWING ---
        # 1. Corner Brackets (Box ki jagah)

        cv2.line(frame, (left, top), (left + 40, top), box_color, 2)
        cv2.line(frame, (left, top), (left, top + 40), box_color, 2)
        cv2.line(frame, (right, top), (right - 40, top), box_color, 2)
        cv2.line(frame, (right, top), (right, top + 40), box_color, 2)
        cv2.line(frame, (left, bottom), (left + 40, bottom), box_color, 2)
        cv2.line(frame, (left, bottom), (left, bottom - 40), box_color, 2)
        cv2.line(frame, (right, bottom), (right - 40, bottom), box_color, 2)
        cv2.line(frame, (right, bottom), (right, bottom - 40), box_color, 2)

        # 2. Scanning Line Animation
        scanner_pos = top + (int(time.time() * 80) % (bottom - top))
        cv2.line(frame, (left, scanner_pos),
                 (right, scanner_pos), (0, 255, 255), 1)

        # Name Display
        # font_scale=0.7 if box_color(0,255,0) else 0.4
        cv2.putText(frame, name.upper(), (left, top - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, box_color, 2)

        # 3. Known person ki details ke liye Glassy Sidebar
        if name in user_details:
            # Semi-transparent background panel
            overlay = frame.copy()
            cv2.rectangle(overlay, (right + 20, top - 20),
                          (right + 450, top + 250), (10, 10, 10), -1)
            cv2.addWeighted(overlay, 0.5, frame, 0.5, 0, frame)
            # Panel border
            cv2.rectangle(frame, (right + 20, top - 20),
                          (right + 450, top + 250), (255, 255, 0), 1)

            # Verification Status
            cv2.putText(frame, "STATUS: ID VERIFIED", (right + 30, top - 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

            details = user_details[name]
            y_offset = top + 25
            for key, value in details.items():
                val_str = ", ".join(value) if isinstance(
                    value, list) else str(value)
                # Label
                cv2.putText(frame, f"[{key.upper()}]", (right + 40, y_offset),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255, 255, 255), 1)
                # Value
                cv2.putText(frame, val_str, (right + 40, y_offset + 18),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.43, (0, 255, 255), 1)
                y_offset += 45

    # Window title
    cv2.imshow('JARVIS-Identity Identification System', frame)

    # Space bar to quit
    if cv2.waitKey(1) & 0xFF == ord(' '):
        break

video_capture.release()
cv2.destroyAllWindows()

HUD System Booting Up...
